# Party City | Alt Data Case Study

### Logan Chalifour

January 12, 2026

# Setup

## Import Packages

First, we install and import packages needed for our analysis.  

In [1]:
# Install packages
'''
!pip install --upgrade pip
!pip install openpyxl
'''

# Imports
import pandas as pd 
import numpy as np
import plotly.graph_objects as go
import re
from sklearn.preprocessing import MinMaxScaler

## ETL

We load the data from the four provided files in and begin investigating.

In [2]:
df_reported = pd.read_excel("reported_numbers.xlsx")
df_monthly_location_sales = pd.read_csv("transaction_data_monthly_sales_by_locationid.csv")
df_location_info = pd.read_csv("transaction_data_location_info.csv")
df_webscrape_locations = pd.read_csv("webscrape_data_locations.csv")

In [3]:
df_reported

,entity,metric,period_end_dt,reported_yoy
0,PRTY,"Brand Comparable Sales Growth, %",2016-03-31,-0.015
1,PRTY,"Brand Comparable Sales Growth, %",2016-06-30,0.038
2,PRTY,"Brand Comparable Sales Growth, %",2016-09-30,0.012
3,PRTY,"Brand Comparable Sales Growth, %",2016-12-31,-0.035
4,PRTY,"Brand Comparable Sales Growth, %",2017-03-31,0.017
5,PRTY,"Brand Comparable Sales Growth, %",2017-06-30,0.001
6,PRTY,"Brand Comparable Sales Growth, %",2017-09-30,-0.026
7,PRTY,"Brand Comparable Sales Growth, %",2017-12-31,-0.014
8,PRTY,"Brand Comparable Sales Growth, %",2018-03-31,0.024
9,PRTY,"Brand Comparable Sales Growth, %",2018-06-30,0.001


In [4]:
df_monthly_location_sales

,symbol,locationid,period_start,period_end,label,amount
0,NYSE:PRTY,1,2019-08-01,2019-08-31,2019-MS08,2128.213469
1,NYSE:PRTY,16,2019-08-01,2019-08-31,2019-MS08,676.387327
2,NYSE:PRTY,17,2019-08-01,2019-08-31,2019-MS08,770.193862
3,NYSE:PRTY,2,2019-08-01,2019-08-31,2019-MS08,2269.441296
4,NYSE:PRTY,21,2019-08-01,2019-08-31,2019-MS08,265.600689
...,...,...,...,...,...,...
38862,NYSE:PRTY,994,2023-12-01,2023-12-31,2023-MS12,987.590765
38863,NYSE:PRTY,995,2023-12-01,2023-12-31,2023-MS12,556.119912
38864,NYSE:PRTY,996,2023-12-01,2023-12-31,2023-MS12,692.716506
38865,NYSE:PRTY,997,2023-12-01,2023-12-31,2023-MS12,1011.097667


In [5]:
df_location_info

,locationid,est_open_date,est_close_date
0,1,2017-01-01 00:00:00.000000,2024-09-01 00:00:00.000000
1,16,2017-01-01 00:00:00.000000,2024-09-01 00:00:00.000000
2,17,2017-01-01 00:00:00.000000,2021-08-01 00:00:00.000000
3,18,2021-05-01 00:00:00.000000,2022-12-01 00:00:00.000000
4,2,2017-01-01 00:00:00.000000,2024-09-01 00:00:00.000000
...,...,...,...
893,994,2018-09-01 00:00:00.000000,2024-09-01 00:00:00.000000
894,995,2017-01-01 00:00:00.000000,2024-09-01 00:00:00.000000
895,996,2020-12-01 00:00:00.000000,2024-09-01 00:00:00.000000
896,997,2020-10-01 00:00:00.000000,2024-09-01 00:00:00.000000


In [6]:
df_webscrape_locations

,address,as_of_date,city,phone,state,location_id,store_services,title,url,zipcode,postcode,latitude,longitude
0,13477 Middlebelt Road,2024-10-07,Livonia,(734) 666-3019,Michigan,539,In-Store Shopping | In-Store Pickup | Curbside...,Livonia Commons,https://stores.partycity.com/us/mi/livonia/par...,48150,48150,42.381156,-83.335770
1,5114 28th Street SE,2024-10-07,Grand Rapids,(616) 365-5419,Michigan,4107,In-Store Shopping | In-Store Pickup | Curbside...,Waterfall Shoppes,https://stores.partycity.com/us/mi/grandrapids...,49512,49512,42.910841,-85.540782
2,2677 Oak Valley Dr,2024-10-07,Ann Arbor,(734) 519-5591,Michigan,4111,In-Store Shopping | In-Store Pickup | Curbside...,Oak Valley Plaza,https://stores.partycity.com/us/mi/annarbor/pa...,48103,48103,42.247490,-83.768959
3,3000 White Bear Avenue North,2024-10-07,Maplewood,(612) 428-0697,Minnesota,1138,In-Store Shopping | In-Store Pickup | Curbside...,Plaza 3000 Shopping Center,https://stores.partycity.com/us/mn/maplewood/p...,55109,55109,45.032238,-93.014647
4,2560 Lemay Ferry Road,2024-10-07,Saint Louis,(314) 396-2245,Missouri,5163,In-Store Shopping | In-Store Pickup | Curbside...,Lemay Plaza,https://stores.partycity.com/us/mo/saintlouis/...,63125,63125,38.518385,-90.305049
...,...,...,...,...,...,...,...,...,...,...,...,...,...
742,2148 North 2nd Street,2024-10-07,Millville,(856) 776-7592,New Jersey,715,In-Store Shopping | In-Store Pickup | Curbside...,Union Lake Crossing,https://stores.partycity.com/us/nj/millville/p...,8332,8332,39.411348,-75.040641
743,530 Consumer Square,2024-10-07,Mays Landing,(609) 829-3193,New Jersey,480,In-Store Shopping | In-Store Pickup | Curbside...,Consumer Square,https://stores.partycity.com/us/nj/mayslanding...,8330,8330,39.452156,-74.632449
744,"14458 Delaware St., Suite 600",2024-10-07,"Suite 600, Westminster",(720) 709-3847,Colorado,1072,In-Store Shopping | In-Store Pickup | Curbside...,Orchard Town Center,https://stores.partycity.com/us/co/westminster...,80023,80023,39.959896,-104.989771
745,2750 Carl T Jones Drive SE,2024-10-07,Huntsville,(256) 701-8153,Alabama,1039,In-Store Shopping | In-Store Pickup | Curbside...,Valley Bend,https://stores.partycity.com/us/al/huntsville/...,35802,35802,34.674648,-86.544348


In [7]:
"""
df_reported_sales.describe()
df_monthly_location_sales.describe()
df_location_info.describe() 
df_webscrape_locations.describe() 
"""

'\ndf_reported_sales.describe()\ndf_monthly_location_sales.describe()\ndf_location_info.describe() \ndf_webscrape_locations.describe() \n'

We convert some fields to datetime for later.

In [8]:
# Convert to datetimes
df_reported["period_end_dt"] = pd.to_datetime(df_reported["period_end_dt"])
df_monthly_location_sales["period_start"] = pd.to_datetime(df_monthly_location_sales["period_start"])
df_monthly_location_sales["period_end"] = pd.to_datetime(df_monthly_location_sales["period_end"])
df_location_info["est_open_date"] = pd.to_datetime(df_location_info["est_open_date"])
df_location_info["est_close_date"] = pd.to_datetime(df_location_info["est_close_date"])


Let's also check for duplicate `locationid`s in `df_location_info`.

In [9]:
df_location_info['locationid'].value_counts()

locationid
18      3
3       3
22      2
320     2
15      2
       ..
338     1
3409    1
346     1
348     1
998     1
Name: count, Length: 866, dtype: int64

There are a few reasons we might see the same `locationid` represented more than once in this dataset. For example, a store could have closed and later reopened under the same ID, or the file may contain updated/corrected open/close dates added as new records over time. We’ll want to validate what these duplicates represent before joining to the transaction data so we don’t inadvertently double-count sales or misclassify stores in the comparable set.

# 1\. Q4 2022 Comparable Sales Outlook

To find the forecasted same store sales for PRTY in Q4'22, we need to find the stores that are open from the beginning of Q4'21 to the end of Q4'22. We create a function `qualifies_as_comparable_store` to be used.

In [10]:
def qualifies_as_comparable_store(df, window_start, window_end):
    """
    Determines whether a store qualifies as a comparable store
    by being open for the full comparison window.
    """
    return (
        (df["est_open_date"] <= window_start) &
        (df["est_close_date"].isna() | (df["est_close_date"] >= window_end))
    )

After defining the forecast windows, we identify the comparable store set by selecting only stores that were open throughout both periods.

In [11]:
# Define the forecast target windows
q4_2021_start = pd.Timestamp("2021-10-01")
q4_2021_end   = pd.Timestamp("2021-12-31")
q4_2022_start = pd.Timestamp("2022-10-01")
q4_2022_end   = pd.Timestamp("2022-12-31")

# Determine the comparable-store set (store must be open in both periods above) 
df_location_info["covers_q4_2021"] = qualifies_as_comparable_store(df_location_info, q4_2021_start, q4_2021_end)
df_location_info["covers_q4_2022"] = qualifies_as_comparable_store(df_location_info, q4_2022_start, q4_2022_end)

df_location_info

,locationid,est_open_date,est_close_date,covers_q4_2021,covers_q4_2022
0,1,2017-01-01,2024-09-01,True,True
1,16,2017-01-01,2024-09-01,True,True
2,17,2017-01-01,2021-08-01,False,False
3,18,2021-05-01,2022-12-01,True,False
4,2,2017-01-01,2024-09-01,True,True
...,...,...,...,...,...
893,994,2018-09-01,2024-09-01,True,True
894,995,2017-01-01,2024-09-01,True,True
895,996,2020-12-01,2024-09-01,True,True
896,997,2020-10-01,2024-09-01,True,True


Let's now find a list of `locationid`s using the location open and close dates to help filter for the panel.

In [12]:
# Locationids can appear multiple times (multiple lifecycle rows / revisions). For panel eligibility, we treat a location as eligible if ANY row supports coverage (max).
df_comp_locations = df_location_info.groupby("locationid", as_index=False)[["covers_q4_2021", "covers_q4_2022"]].max()

# If both are True, then True
df_comp_locations["is_comp_q4_2022"] = df_comp_locations["covers_q4_2021"] & df_comp_locations["covers_q4_2022"]

# Get list of locationids for panel
comp_ids = set(df_comp_locations.loc[df_comp_locations["is_comp_q4_2022"], "locationid"].tolist())

We filter the monthly location sales data for those that are part of the panel and aggregate by quarter.

In [13]:
# Calibrate sample YoY to reported YoY (to match reported definition)
df_comp_monthly_location_sales = df_monthly_location_sales[df_monthly_location_sales["locationid"].isin(comp_ids)].copy()

# Bucket monthly rows into quarter-end dates
df_comp_monthly_location_sales["quarter_end"] = df_comp_monthly_location_sales["period_end"].dt.to_period("Q").dt.end_time.dt.normalize()

# Aggregate by quarter
df_sample_comp_hist_qtr = (
    df_comp_monthly_location_sales.groupby("quarter_end", as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "sample_comp_sales"})
    .sort_values("quarter_end")
)

df_sample_comp_hist_qtr["sample_yoy"] = df_sample_comp_hist_qtr["sample_comp_sales"].pct_change(4)

df_sample_comp_hist_qtr

,quarter_end,sample_comp_sales,sample_yoy
0,2019-09-30,2.499795e+06,NaN
1,2019-12-31,5.730950e+06,NaN
2,2020-03-31,2.832172e+06,NaN
3,2020-06-30,2.195535e+06,NaN
4,2020-09-30,4.004437e+06,0.601906
5,2020-12-31,5.422557e+06,-0.053812
6,2021-03-31,3.681907e+06,0.300030
7,2021-06-30,5.005860e+06,1.280018
8,2021-09-30,4.387902e+06,0.095760
9,2021-12-31,6.282300e+06,0.158549


We now join in the the actual quarterly sales figures to compare to the sample.

In [14]:
df_actual_and_sample = (
    df_reported
        .rename(columns={"period_end_dt": "quarter_end"})
        .merge(
            df_sample_comp_hist_qtr,
            on="quarter_end",
            how="outer"
        )
)

df_actual_and_sample

,entity,metric,quarter_end,reported_yoy,sample_comp_sales,sample_yoy
0,PRTY,"Brand Comparable Sales Growth, %",2016-03-31,-0.015,NaN,NaN
1,PRTY,"Brand Comparable Sales Growth, %",2016-06-30,0.038,NaN,NaN
2,PRTY,"Brand Comparable Sales Growth, %",2016-09-30,0.012,NaN,NaN
3,PRTY,"Brand Comparable Sales Growth, %",2016-12-31,-0.035,NaN,NaN
4,PRTY,"Brand Comparable Sales Growth, %",2017-03-31,0.017,NaN,NaN
5,PRTY,"Brand Comparable Sales Growth, %",2017-06-30,0.001,NaN,NaN
6,PRTY,"Brand Comparable Sales Growth, %",2017-09-30,-0.026,NaN,NaN
7,PRTY,"Brand Comparable Sales Growth, %",2017-12-31,-0.014,NaN,NaN
8,PRTY,"Brand Comparable Sales Growth, %",2018-03-31,0.024,NaN,NaN
9,PRTY,"Brand Comparable Sales Growth, %",2018-06-30,0.001,NaN,NaN


Now let's plot the relationship between `sample_yoy` growth and `reported_yoy` growth with a line graph to understand how tightly coupled they are.

In [15]:
fig = go.Figure()

# Reported YoY line
fig.add_trace(
    go.Scatter(
        x = df_actual_and_sample["quarter_end"],
        y = df_actual_and_sample["reported_yoy"] * 100,
        mode = "lines+markers",
        name = "Reported Comparable Sales YoY",
        line = dict(width=3)
    )
)

# Sample YoY line
fig.add_trace(
    go.Scatter(
        x = df_actual_and_sample["quarter_end"],
        y = df_actual_and_sample["sample_yoy"] * 100,
        mode = "lines+markers",
        name = "Sample-Based Comparable Sales YoY",
        line = dict(width=3, dash="dash")
    )
)

fig.update_layout(
    title = "Reported vs Sample-Based Comparable Sales YoY (Party City)",
    xaxis_title = "Quarter",
    yaxis_title = "YoY Growth (%)",
    template = "plotly_white",
    legend_title = "Series",
    hovermode = "x unified"
)

fig.show()


Given the strong historical relationship between the sample-based comparable sales growth and Party City's reported figures, we use the sample signal as a proxy for Q4’22 performance. Based on this approach, we estimate Q4 2022 Brand Comparable Sales YoY growth of approximately **−9.9%**.

# 2\. Location Score Rankings

First, we clean `df_webscrape_locations` to make it more suitable for our analysis.

In [16]:
# Copy dataframe to preserve original
df_web = df_webscrape_locations.copy()

# Remove empty spaces
for c in ["address", "city", "state", "store_services"]:
    if c in df_web.columns:
        df_web[c] = df_web[c].astype(str).str.strip()

# Capitalize states
if "state" in df_web.columns:
    df_web["state"] = df_web["state"].str.upper()

# Build list of unique services
services_list = (
    df_web["store_services"]
        .str.split(r",|\|", regex=True)
        .apply(lambda lst: [s.strip() for s in lst if s and s.strip()])
)
unique_services = sorted({s for sublist in services_list for s in sublist})

# Create binary columns for each service
for svc in unique_services:
    col_name = f"svc_{svc.replace(' ', '_')}"
    df_web[col_name] = df_web["store_services"].str.contains(
        re.escape(svc),
        na=False
    ).astype(int)

# Collapse duplicate helium service columns for svc_helium + svc_helium_inflation
if ("svc_Helium" in df_web.columns) or ("svc_Helium_Inflation" in df_web.columns):
    df_web["svc_Helium"] = (
        df_web.get("svc_Helium", 0).astype(int) |
        df_web.get("svc_Helium_Inflation", 0).astype(int)
    ).astype(int)

    df_web.drop(columns=["svc_Helium_Inflation"], inplace=True, errors="ignore")

# Service count (not including flaship indicator)
svc_cols = [
    c for c in df_web.columns
    if c.startswith("svc_") and "flag" not in c.lower()
]
df_web["service_count"] = df_web[svc_cols].sum(axis=1)

df_web

,address,as_of_date,city,phone,state,location_id,store_services,title,url,zipcode,...,svc_Balloon_Delivery,svc_Curbside_Pickup,svc_Delivery,svc_Flag_Ship_Store,svc_Helium,svc_In-Store_Pickup,svc_In-Store_Shopping,svc_Next_Gen,svc_Scheduled_or_Same_Day_Delivery,service_count
0,13477 Middlebelt Road,2024-10-07,Livonia,(734) 666-3019,MICHIGAN,539,In-Store Shopping | In-Store Pickup | Curbside...,Livonia Commons,https://stores.partycity.com/us/mi/livonia/par...,48150,...,1,1,1,0,1,1,1,0,1,7
1,5114 28th Street SE,2024-10-07,Grand Rapids,(616) 365-5419,MICHIGAN,4107,In-Store Shopping | In-Store Pickup | Curbside...,Waterfall Shoppes,https://stores.partycity.com/us/mi/grandrapids...,49512,...,1,1,1,0,1,1,1,0,1,7
2,2677 Oak Valley Dr,2024-10-07,Ann Arbor,(734) 519-5591,MICHIGAN,4111,In-Store Shopping | In-Store Pickup | Curbside...,Oak Valley Plaza,https://stores.partycity.com/us/mi/annarbor/pa...,48103,...,1,1,1,0,1,1,1,0,1,7
3,3000 White Bear Avenue North,2024-10-07,Maplewood,(612) 428-0697,MINNESOTA,1138,In-Store Shopping | In-Store Pickup | Curbside...,Plaza 3000 Shopping Center,https://stores.partycity.com/us/mn/maplewood/p...,55109,...,1,1,1,0,1,1,1,0,1,7
4,2560 Lemay Ferry Road,2024-10-07,Saint Louis,(314) 396-2245,MISSOURI,5163,In-Store Shopping | In-Store Pickup | Curbside...,Lemay Plaza,https://stores.partycity.com/us/mo/saintlouis/...,63125,...,1,1,1,0,1,1,1,0,1,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
742,2148 North 2nd Street,2024-10-07,Millville,(856) 776-7592,NEW JERSEY,715,In-Store Shopping | In-Store Pickup | Curbside...,Union Lake Crossing,https://stores.partycity.com/us/nj/millville/p...,8332,...,1,1,1,0,1,1,1,0,1,7
743,530 Consumer Square,2024-10-07,Mays Landing,(609) 829-3193,NEW JERSEY,480,In-Store Shopping | In-Store Pickup | Curbside...,Consumer Square,https://stores.partycity.com/us/nj/mayslanding...,8330,...,1,1,1,0,1,1,1,0,1,7
744,"14458 Delaware St., Suite 600",2024-10-07,"Suite 600, Westminster",(720) 709-3847,COLORADO,1072,In-Store Shopping | In-Store Pickup | Curbside...,Orchard Town Center,https://stores.partycity.com/us/co/westminster...,80023,...,1,1,1,0,1,1,1,0,1,7
745,2750 Carl T Jones Drive SE,2024-10-07,Huntsville,(256) 701-8153,ALABAMA,1039,In-Store Shopping | In-Store Pickup | Curbside...,Valley Bend,https://stores.partycity.com/us/al/huntsville/...,35802,...,1,1,1,0,1,1,1,0,1,7


Then, we join TTM sales for all stores to get an idea of how much each is selling. 

In [17]:
# Identify the most recent month in the data
latest_month = df_monthly_location_sales["period_end"].max()
ttm_start = latest_month - pd.DateOffset(months=12)

# Keep only the trailing 12 months
df_ttm = df_monthly_location_sales[df_monthly_location_sales["period_end"] > ttm_start].copy()

# Aggregate TTM sales per store
df_ttm_sales_per_store = (
    df_ttm
        .groupby("locationid", as_index=False)["amount"]
        .sum()
        .rename(columns={"amount": "ttm_sales"})
)

# Keep all web-scraped stores, attach TTM sales where available
df_web = (
    df_web
        .merge(
            df_ttm_sales_per_store,
            left_on="location_id",
            right_on="locationid",
            how="left"
        )
)

df_web

,address,as_of_date,city,phone,state,location_id,store_services,title,url,zipcode,...,svc_Delivery,svc_Flag_Ship_Store,svc_Helium,svc_In-Store_Pickup,svc_In-Store_Shopping,svc_Next_Gen,svc_Scheduled_or_Same_Day_Delivery,service_count,locationid,ttm_sales
0,13477 Middlebelt Road,2024-10-07,Livonia,(734) 666-3019,MICHIGAN,539,In-Store Shopping | In-Store Pickup | Curbside...,Livonia Commons,https://stores.partycity.com/us/mi/livonia/par...,48150,...,1,0,1,1,1,0,1,7,539.0,11026.804360
1,5114 28th Street SE,2024-10-07,Grand Rapids,(616) 365-5419,MICHIGAN,4107,In-Store Shopping | In-Store Pickup | Curbside...,Waterfall Shoppes,https://stores.partycity.com/us/mi/grandrapids...,49512,...,1,0,1,1,1,0,1,7,4107.0,13045.035088
2,2677 Oak Valley Dr,2024-10-07,Ann Arbor,(734) 519-5591,MICHIGAN,4111,In-Store Shopping | In-Store Pickup | Curbside...,Oak Valley Plaza,https://stores.partycity.com/us/mi/annarbor/pa...,48103,...,1,0,1,1,1,0,1,7,4111.0,11134.150727
3,3000 White Bear Avenue North,2024-10-07,Maplewood,(612) 428-0697,MINNESOTA,1138,In-Store Shopping | In-Store Pickup | Curbside...,Plaza 3000 Shopping Center,https://stores.partycity.com/us/mn/maplewood/p...,55109,...,1,0,1,1,1,0,1,7,NaN,NaN
4,2560 Lemay Ferry Road,2024-10-07,Saint Louis,(314) 396-2245,MISSOURI,5163,In-Store Shopping | In-Store Pickup | Curbside...,Lemay Plaza,https://stores.partycity.com/us/mo/saintlouis/...,63125,...,1,0,1,1,1,0,1,7,5163.0,9554.606838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
742,2148 North 2nd Street,2024-10-07,Millville,(856) 776-7592,NEW JERSEY,715,In-Store Shopping | In-Store Pickup | Curbside...,Union Lake Crossing,https://stores.partycity.com/us/nj/millville/p...,8332,...,1,0,1,1,1,0,1,7,715.0,19396.736080
743,530 Consumer Square,2024-10-07,Mays Landing,(609) 829-3193,NEW JERSEY,480,In-Store Shopping | In-Store Pickup | Curbside...,Consumer Square,https://stores.partycity.com/us/nj/mayslanding...,8330,...,1,0,1,1,1,0,1,7,480.0,13221.018273
744,"14458 Delaware St., Suite 600",2024-10-07,"Suite 600, Westminster",(720) 709-3847,COLORADO,1072,In-Store Shopping | In-Store Pickup | Curbside...,Orchard Town Center,https://stores.partycity.com/us/co/westminster...,80023,...,1,0,1,1,1,0,1,7,1072.0,11017.221902
745,2750 Carl T Jones Drive SE,2024-10-07,Huntsville,(256) 701-8153,ALABAMA,1039,In-Store Shopping | In-Store Pickup | Curbside...,Valley Bend,https://stores.partycity.com/us/al/huntsville/...,35802,...,1,0,1,1,1,0,1,7,1039.0,9545.304967


To evaluate which Party City stores are most likely to be retained versus closed, we build a simple and transparent store-level scoring framework. The goal is not to predict exact closures, but to rank stores based on characteristics that typically matter most in retail storefronts.

Each store is scored across four dimensions, with equal weight (25% each) to keep the model interpretable and avoid over-complexity. 

**Flagship status**  
Flagship stores are generally higher-priority locations due to their brand presence and scale. Stores designated as flagships receive a score of 1, while non-flagship stores receive a 0.

**Depth of in-store services**  
Stores offering a broader set of services tend to deliver a stronger customer experience and higher engagement. We use the number of services (excluding "flagship") offered by each store and normalize it to a 0–1 scale so locations can be compared consistently.

**Local market redundancy**  
Retail restructurings often focus on reducing overlapping store coverage. We measure redundancy by counting how many Party City locations exist in the same city and state. Stores in less crowded markets score higher using the inverse of the local store count, while stores in more saturated markets score lower.

**Recent sales performance**  
Where available, we incorporate trailing-twelve-month sales from the transaction sample as a proxy for store productivity. Sales are normalized to a 0–1 scale. For stores without reliable sales data, we assign the average sales score to avoid penalizing them due to data limitations.

The final store score is the average of these four components. Higher-scoring stores represent stronger candidates to retain, while lower-scoring stores are more likely candidates for closure as Party City moves toward a leaner, more profitable store footprint.

In [18]:
# 1) Flagship score: 1 if flagship else 0
df_web["score_flagship"] = df_web["svc_Flag_Ship_Store"].astype(int)

# 2) Service score: normalize service_count to 0-1
scaler = MinMaxScaler()
df_web["score_service"] = scaler.fit_transform(df_web[["service_count"]])

# 3) Market uniqueness score: inverse store count for City, State combination
df_web["stores_in_city_state"] = (
    df_web.groupby(["city", "state"])["location_id"].transform("count")
)
df_web["score_market_unique"] = 1 / df_web["stores_in_city_state"]

# 4) Sales score: normalize ttm_sales to 0-1 and missing sales with the average score
scaler = MinMaxScaler()
df_web["score_sales"] = scaler.fit_transform(df_web[["ttm_sales"]])
df_web["score_sales"] = df_web["score_sales"].fillna(df_web["score_sales"].mean())

# 5) Final score: equal weights (25% each)
df_web["final_location_score"] = 0.25 * (
    df_web["score_flagship"] +
    df_web["score_service"] +
    df_web["score_market_unique"] +
    df_web["score_sales"]
)

# Assign store ranks
df_web["final_location_rank"] = df_web["final_location_score"].rank(ascending=False, method="dense").astype(int)

# Output scored_df
df_scored = df_web[
    [
        "location_id", "address", "city", "state", "store_services",
        "ttm_sales", "service_count", "stores_in_city_state",
        "score_flagship", "score_service", "score_market_unique", "score_sales",
        "final_location_score", "final_location_rank"
    ]
].sort_values("final_location_score", ascending=False).reset_index(drop=True)

df_scored

,location_id,address,city,state,store_services,ttm_sales,service_count,stores_in_city_state,score_flagship,score_service,score_market_unique,score_sales,final_location_score,final_location_rank
0,941,8026 West Bell Road,Glendale,ARIZONA,Flag Ship Store | In-Store Shopping | Next Gen...,82049.809062,8,1,1,1.000000,1.00,0.158214,0.789553,1
1,1095,5600 Urbana Pike,Frederick,MARYLAND,Flag Ship Store | In-Store Shopping | Next Gen...,52158.994365,8,1,1,1.000000,1.00,0.100244,0.775061,2
2,913,2009 Black Rock Turnpike,Fairfield,CONNECTICUT,Flag Ship Store | In-Store Shopping | Next Gen...,40422.501558,8,1,1,1.000000,1.00,0.077482,0.769370,3
3,738,9101 Woodmore Centre Drive,Lanham,MARYLAND,Flag Ship Store | In-Store Shopping | Next Gen...,37836.956887,8,1,1,1.000000,1.00,0.072468,0.768117,4
4,408,1684 Route 22 East,Watchung,NEW JERSEY,Flag Ship Store | In-Store Shopping | Next Gen...,30498.795286,8,1,1,1.000000,1.00,0.058236,0.764559,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
742,364,11150 Research Boulevard,Austin,TEXAS,In-Store Shopping | Helium,13052.452246,2,4,0,0.142857,0.25,0.024401,0.104314,639
743,24,13419 San Pedro Avenue,San Antonio,TEXAS,In-Store Shopping | Helium,7666.893711,2,4,0,0.142857,0.25,0.013956,0.101703,640
744,217,10628 Culebra Road,San Antonio,TEXAS,In-Store Shopping | Helium,6821.310770,2,4,0,0.142857,0.25,0.012316,0.101293,641
745,199,5127 NW Loop 410,San Antonio,TEXAS,In-Store Shopping | Helium,6684.728953,2,4,0,0.142857,0.25,0.012051,0.101227,642


Using the output of the scoring model above, we identify the lowest-scoring locations as the most likely closure candidates. While it is unclear exactly how many stores Party City intends to close, prioritizing locations starting from the bottom of this ranking would be a reasonable approach. Notably, a disproportionate number of these lower-scoring stores are located in Texas, particularly in San Antonio and Austin, suggesting potential market saturation in that region.

In [19]:
df_scored.tail(10)

,location_id,address,city,state,store_services,ttm_sales,service_count,stores_in_city_state,score_flagship,score_service,score_market_unique,score_sales,final_location_score,final_location_rank
737,1241,11465 Carmel Mountain Road,San Diego,CALIFORNIA,In-Store Shopping | In-Store Pickup | Helium,NaN,3,4,0,0.285714,0.250,0.042397,0.144528,635
738,1240,3309 Rosecrans Street,San Diego,CALIFORNIA,In-Store Shopping | In-Store Pickup | Helium,NaN,3,4,0,0.285714,0.250,0.042397,0.144528,635
739,23,9525 Westheimer Street,Houston,TEXAS,In-Store Shopping | In-Store Pickup | Curbside...,11858.551039,4,8,0,0.428571,0.125,0.022085,0.143914,636
740,538,3860 South Maryland Parkway,Las Vegas,NEVADA,In-Store Shopping | Next Gen | Helium,3814.697622,3,4,0,0.285714,0.250,0.006485,0.135550,637
741,363,5601 Brodie Lane,Austin,TEXAS,In-Store Shopping | Helium,32977.030742,2,4,0,0.142857,0.250,0.063042,0.113975,638
742,364,11150 Research Boulevard,Austin,TEXAS,In-Store Shopping | Helium,13052.452246,2,4,0,0.142857,0.250,0.024401,0.104314,639
743,24,13419 San Pedro Avenue,San Antonio,TEXAS,In-Store Shopping | Helium,7666.893711,2,4,0,0.142857,0.250,0.013956,0.101703,640
744,217,10628 Culebra Road,San Antonio,TEXAS,In-Store Shopping | Helium,6821.310770,2,4,0,0.142857,0.250,0.012316,0.101293,641
745,199,5127 NW Loop 410,San Antonio,TEXAS,In-Store Shopping | Helium,6684.728953,2,4,0,0.142857,0.250,0.012051,0.101227,642
746,360,2920 SW Military Drive,San Antonio,TEXAS,In-Store Shopping | Helium,2574.818874,2,4,0,0.142857,0.250,0.004080,0.099234,643


# 3\. Strategic and Secular Considerations

Party City operates in a highly seasonal and discretionary retail category, which creates both opportunity and risk. On the positive side, the company has strong brand recognition and covers a wide range of life events and holidays, which drives recurring demand and makes it a familiar destination for consumers planning celebrations.

At the same time, several structural challenges stand out. Demand is highly cyclical, making the business sensitive to economic slowdowns and inventory missteps. The large brick-and-mortar footprint, which historically supported convenience, has become harder to justify as consumers increasingly shift online and toward big-box or general-merchandise retailers. Party supplies are also largely commoditized, limiting pricing power and making customer loyalty difficult to sustain.

Overall, Party City’s brand and category breadth remain valuable, but long-term success likely depends on a smaller, more productive store footprint, tighter execution around seasonality, and a stronger online presence that better aligns with current consumer behavior.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=d9e136f6-61ec-429e-ac19-8696445300f6' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>